In [1]:
import pandas as pd, numpy as np, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns, warnings, pickle, os
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (r2_score, mean_absolute_error, mean_squared_error,
                             classification_report, roc_auc_score, confusion_matrix)
from sklearn.preprocessing import LabelEncoder

In [2]:
df = pd.read_csv('../data/raw/india_agriculture_seed_sales_data.csv')

In [3]:

df['Date'] = pd.to_datetime(df['Date'], format='%Y-%m')
df['Price_Cost_Ratio'] = df['Seed_Price_Per_Unit'] / df['Cost_Per_Unit']

In [4]:
# Encoding 
le_region = LabelEncoder(); le_veg = LabelEncoder()
df['Region_enc'] = le_region.fit_transform(df['Region'])
df['Veg_enc']    = le_veg.fit_transform(df['Vegetable_Type'])

In [5]:
FEATURES = [
    'Year','Month','Region_enc','Veg_enc',
    'Warehouse_Quantity','Competitor_Market_Share_%','Company_Market_Share_%',
    'Farmer_Sentiment_Score','Soil_pH','Soil_Nitrogen_ppm',
    'Soil_Phosphorus_ppm','Soil_Potassium_ppm','Rainfall_mm',
    'Temperature_C','Irrigation_Access_%','Pest_Infestation_Index',
    'Seed_Quality_Index_%','Seed_Price_Per_Unit','Cost_Per_Unit',
    'Wholesale_Price','Retail_Price','Farmer_Share_%','Price_Cost_Ratio'
]
X = df[FEATURES]
print('=== ML MODEL TRAINING ===')
print(f'Records: {len(X):,}  |  Features: {len(FEATURES)}')

=== ML MODEL TRAINING ===
Records: 20,000  |  Features: 23


In [6]:
# MODEL 1: Profit Margin Predictor 

print('\n[1/3] Profit Margin Predictor (Random Forest)...')
y1 = df['Profit_Margin_%']
X1tr,X1te,y1tr,y1te = train_test_split(X,y1,test_size=0.2,random_state=42)
rf = RandomForestRegressor(n_estimators=300,max_depth=14,min_samples_leaf=4,random_state=42,n_jobs=-1)
rf.fit(X1tr, y1tr)
y1p  = rf.predict(X1te)
r2pm = r2_score(y1te, y1p)
maepm= mean_absolute_error(y1te, y1p)
cvpm = cross_val_score(rf, X, y1, cv=5, scoring='r2', n_jobs=-1).mean()
print(f'  Test R²={r2pm:.4f}  |  MAE={maepm:.4f}%  |  5-fold CV R²={cvpm:.4f}')
imp1 = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=False)
print(f'  Top 5 features: {list(imp1.head(5).index)}')


[1/3] Profit Margin Predictor (Random Forest)...
  Test R²=1.0000  |  MAE=0.0003%  |  5-fold CV R²=0.9555
  Top 5 features: ['Price_Cost_Ratio', 'Rainfall_mm', 'Soil_Phosphorus_ppm', 'Soil_Potassium_ppm', 'Temperature_C']


In [7]:
# MODEL 2: Revenue Growth Predictor
print('\n[2/3] Revenue Growth % Predictor (Gradient Boosting)...')
y2 = df['Revenue_Growth_%']
X2tr,X2te,y2tr,y2te = train_test_split(X,y2,test_size=0.2,random_state=42)
gb = GradientBoostingRegressor(n_estimators=200,max_depth=7,learning_rate=0.08,subsample=0.8,random_state=42)
gb.fit(X2tr, y2tr)
y2p  = gb.predict(X2te)
r2rg = r2_score(y2te, y2p)
maerg= mean_absolute_error(y2te, y2p)
print(f'  Test R²={r2rg:.4f}  |  MAE={maerg:.4f}%')
imp2 = pd.Series(gb.feature_importances_, index=FEATURES).sort_values(ascending=False)


[2/3] Revenue Growth % Predictor (Gradient Boosting)...
  Test R²=0.9339  |  MAE=1.3110%


In [8]:
# MODEL 3: Churn Risk Classifier
print('\n[3/3] Churn Risk Classifier (Random Forest)...')
q25s = df['Company_Market_Share_%'].quantile(0.25)
q25f = df['Farmer_Sentiment_Score'].quantile(0.25)
df['Churn_Risk'] = ((df['Company_Market_Share_%']<q25s) & (df['Farmer_Sentiment_Score']<q25f)).astype(int)
print(f'  Label distribution: {df["Churn_Risk"].value_counts().to_dict()}')
y3 = df['Churn_Risk']
X3tr,X3te,y3tr,y3te = train_test_split(X,y3,test_size=0.2,stratify=y3,random_state=42)
rfc = RandomForestClassifier(n_estimators=200,max_depth=10,random_state=42,n_jobs=-1)
rfc.fit(X3tr, y3tr)
y3p = rfc.predict(X3te); y3pb = rfc.predict_proba(X3te)[:,1]
auc = roc_auc_score(y3te, y3pb)
print(f'  AUC={auc:.4f}')
print(classification_report(y3te, y3p))



[3/3] Churn Risk Classifier (Random Forest)...
  Label distribution: {0: 18770, 1: 1230}
  AUC=1.0000
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      3754
           1       1.00      1.00      1.00       246

    accuracy                           1.00      4000
   macro avg       1.00      1.00      1.00      4000
weighted avg       1.00      1.00      1.00      4000



In [9]:
# Save models
with open('../outputs/Models/profit_margin_model.pkl','wb') as f: pickle.dump(rf, f)
with open('../outputs/Models/revenue_growth_model.pkl','wb') as f: pickle.dump(gb, f)
with open('../outputs/Models/churn_risk_model.pkl','wb') as f: pickle.dump(rfc, f)


In [10]:
# FIGURE 9: Model Performance 
fig = plt.figure(figsize=(20,13))
gs  = gridspec.GridSpec(2,3, figure=fig, hspace=0.48, wspace=0.38)

idx = np.random.choice(len(y1te),min(2000,len(y1te)),replace=False)

ax1 = fig.add_subplot(gs[0,0])
ax1.scatter(y1te.iloc[idx], y1p[idx], alpha=0.3, s=8, color='steelblue')
lims=[y1te.min(),y1te.max()]; ax1.plot(lims,lims,'r--',lw=1.5)
ax1.set_xlabel('Actual Margin %'); ax1.set_ylabel('Predicted %')
ax1.set_title(f'Model 1: Profit Margin\nR²={r2pm:.4f} | MAE={maepm:.4f}%')

ax2 = fig.add_subplot(gs[0,1])
imp1.head(14).plot(kind='barh', ax=ax2, color='steelblue')
ax2.set_title('Feature Importance — Profit Margin Model')

ax3 = fig.add_subplot(gs[0,2])
ax3.scatter(y2te.iloc[idx], y2p[idx], alpha=0.3, s=8, color='#2ca02c')
lims2=[y2te.min(),y2te.max()]; ax3.plot(lims2,lims2,'r--',lw=1.5)
ax3.set_xlabel('Actual Growth %'); ax3.set_ylabel('Predicted %')
ax3.set_title(f'Model 2: Revenue Growth\nR²={r2rg:.4f} | MAE={maerg:.4f}%')

ax4 = fig.add_subplot(gs[1,0])
imp2.head(14).plot(kind='barh', ax=ax4, color='#2ca02c')
ax4.set_title('Feature Importance — Revenue Growth Model')

ax5 = fig.add_subplot(gs[1,1])
cm = confusion_matrix(y3te, y3p)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax5,
            xticklabels=['No Churn','Churn'], yticklabels=['No Churn','Churn'])
ax5.set_title(f'Model 3: Churn Risk Classifier\nAUC={auc:.4f}')
ax5.set_xlabel('Predicted'); ax5.set_ylabel('Actual')

ax6 = fig.add_subplot(gs[1,2])
residuals = y1te.values - y1p
ax6.hist(residuals, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
ax6.axvline(0, color='red', lw=1.5, ls='--', label='Zero error')
ax6.axvline(residuals.mean(), color='orange', lw=1.5, ls='--',
            label=f'Mean={residuals.mean():.3f}')
ax6.set_xlabel('Residual (Actual − Predicted)'); ax6.set_ylabel('Count')
ax6.set_title('Profit Margin — Residual Distribution'); ax6.legend(fontsize=8)

fig.suptitle('GREEN HARVEST SEEDS — ML Model Performance Summary',
             fontsize=13, fontweight='bold', y=1.01)
plt.savefig('../outputs/figures/ml_models.png', bbox_inches='tight', dpi=140)
print('Saved: 09_ml_models.png')
plt.show()


Saved: 09_ml_models.png
